# PRÁCTICA BACKTESTING AVANZADO

##### Realizado por Mateo Santos

In [1]:
# Importamos las librerías
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import pyarrow
import pyarrow.parquet as pq
import time

# Fijamos la semilla
np.random.seed(42)

In [2]:
# Definición de variables globales
capital = 250000
#START_DATE = '2015-01-01'
START_DATE = '2014-01-01'
END_DATE = '2026-01-31'
NUM_ACTIVOS = 20
capital_por_activo = capital/NUM_ACTIVOS

## Notebook 3: Implementación de la estrategia

### 3.1. Paso A: Retorno Acumulado con "Lag" de 1 Mes

La función **`get_momentum_raw`** calcula el momentum acumulado durante los últimos **$r$** meses a partir de los retornos con un lag de un mes.

Gracias a las propiedades aditivas de **retornos logarítmicos**, el valor del momentum se calcula como la **suma de los retornos desde el mes r-13 hasta el mes r-1**. Para el cálculo del momentum no se tienen en cuenta los retornos del mes actual.

In [18]:
# Función para calcular el momentum durante r meses sin contar el actual. (Desde t-1 hasta t-r-1)
def get_momentum_raw(returns, r):
    # Obtenemos la suma de los retornos de cada mes. La suma de los retornos logarítmicos es la rentabilidad acumulada mensual
    df = returns.resample('BME').sum()
    # Realizamos la agregación en función del número de periodos. 
    # Como cada mes tiene la rentabilidad acumulada. El momentum se calcula como la suma de estas rentabilidades logarítmicas
    # acumuladas
    df = df.rolling(window=r).sum()
    # Tenemos en cuenta los meses desde t-1 hasta t-r-1
    return df.shift(1).iloc[r:]

In [19]:
# Obtenemos los momentum de 6 meses y 12 meses para cada activo y mes
r6 = get_momentum_raw(returns_close, 6)
r12 = get_momentum_raw(returns_close, 12)

### 3.2. Paso B: Normalización por Factor (Z-Score)

La función **`get_mean_std_period`** obtiene para cada fecha la media y desviación típica de los momentum de todos los activos. Es necesario para normalizar los momentum y que sean comparables.

In [20]:
# Función para obtener la media y desviación típica de cada mes del momentum
def get_mean_std_period(r):
    r_mean = r.mean(axis = 1).values.reshape(-1,1)
    r_std = r.std(axis = 1).values.reshape(-1,1)
    return r_mean, r_std

In [21]:
# Calculamos la media y desviación del momentum para cada periodo
r6_mean, r6_std = get_mean_std_period(r6)
r12_mean, r12_std = get_mean_std_period(r12)

La función **`get_Z_values`** obtiene los valores Z de cada uno de los momentum (de 6 meses y de 12 meses). Los valores Z corresponden a la **normalización de los momentum** para que sean comparables.

Cada valor de la matriz de momentum se normaliza en función de la media y la desviación de los momentum de cada fecha tal que:
$Z = \frac{r - \mu_{r}}{\sigma_{r}}$.

Gracias a esta normalización, ahora sí que podemos comparar los momentum.

In [22]:
# Función para estandarizar los momentum
# Para cada momentum le restamos su propia media y dividimos por su desviación típica, respetando los índices
def get_Z_values(r, mu, sigma):
    return (r - mu)/sigma

In [23]:
# Seleccionamos a partir de la sexta fila para que coincidan los índices
z6 = get_Z_values(r6, r6_mean, r6_std).iloc[6:]
z12 = get_Z_values(r12, r12_mean, r12_std)

Filtramos por el último día bursátil y hábil de cada mes. Es clave porque esas fechas constituyen las de rebalanceo almacenadas en `fechas_reales`.

In [24]:
# Fijamos los índices que usaremos para identificar cada periodo (último día hábil de cada mes)
# Filtramos por los precios del periodo de estudio (a partir de enero de 2015)
# Seleccionamos solo los precios del día del rebalanceo para poder hacer las compras y ventas de activos
indices_rebalanceo = z12.index

# Obtenemos las posiciones de los índices de rebalanceo en el dataset de precios
# 'pad' asegura que si la fecha no existe, busque la última fecha real anterior
indices_validos_pos = precios_close.index.get_indexer(indices_rebalanceo, method='pad')

# Extraemos las fechas REALES que existen en el mercado
fechas_reales = precios_close.index[indices_validos_pos]

### 3.3. Paso C: Puntuación Compuesta y Selección

La función **`get_Z`** calcula la media simple de ambos Z-Score normalizados tal que $Z = \frac{Z_6 + Z_{12}}{2}$.

In [25]:
# El score final se obtiene como la media de los momentum 6 y 12 estandarizados
def get_Z(z_x, z_y):
    return (z_x + z_y)/2

In [26]:
# Obtenemos los Z-Scores finales que nos servirán para identificar los activos en los que hay que invertir
z = get_Z(z6, z12)
z.index = fechas_reales

Finalmente, gracias a la función **`select_assets`** seleccionamos para cada fecha invertir en los activos válidos que tengan un mayor valor Z.

Es clave la función `rank` que permite **seleccionar los activos con mayor valor Z-Score en cada fecha** y filtrar por la variable **`activos_aptos`**, que es la matriz booleana que determina en que periodos se puede invertir en cada activo.

Es clave para cada activo, que no se pueda volver a seleccionar una vez ha dejado de cotizar.

In [27]:
# Generamos una matriz booleana para cada periodo en el que asigna el valor True a los 20 activos con mayor Z-score
# Los valores booleanos True pueden actuar como valor 1. Por lo que ya tenemos la matriz binaria de pesos
def select_assets(z, n_activos = 20):
    return z.rank(axis=1, ascending=False) <= n_activos

In [28]:
# 1. Calculamos el último día de cotización
ultimo_dia_cotizacion = precios_close.notna()[::-1].idxmax()

# 2. Creamos la máscara de disponibilidad
disponibilidad_real = activos_aptos.copy()

# 3. Iteramos solo sobre activos que existan en ambos objetos
activos_comunes = disponibilidad_real.columns.intersection(ultimo_dia_cotizacion.index)

for activo in activos_comunes:
    # Aplicamos la lógica de fecha límite
    fecha_limite = ultimo_dia_cotizacion[activo]
    disponibilidad_real[activo] = disponibilidad_real[activo] & (disponibilidad_real.index < fecha_limite)

# 4. (Opcional) Si hay activos en disponibilidad_real que NO estaban en precios_close, 
# por seguridad los ponemos a False ya que no tenemos datos de su delisting
activos_sin_datos = disponibilidad_real.columns.difference(ultimo_dia_cotizacion.index)
if not activos_sin_datos.empty:
    disponibilidad_real[activos_sin_datos] = False
    print(f"Advertencia: {len(activos_sin_datos)} activos no tenían datos de precio y se han marcado como no disponibles.")

# --- PASO B: Filtrar Z-Score ---
# Aplicamos la máscara: si no está disponible, el Z-score pasa a ser NaN
z_final = z.where(disponibilidad_real)

# --- PASO C: Selección de activos ---
# Ahora rank() solo verá activos que REALMENTE cotizan. 
# Si el top 20 original tenía un delisting, el que estaba en el puesto 21 subirá al 20.
select_assets_period = select_assets(z_final, n_activos = NUM_ACTIVOS) * 1

# Aseguramos fechas
select_assets_period.index = fechas_reales

Advertencia: 437 activos no tenían datos de precio y se han marcado como no disponibles.


Generamos el fichero csv con los activos seleccionados a invertir en cada periodo.

In [29]:
# Las claves del diccionario son las fechas de rebalanceo y los valores los activos en los que se invierte
resultado = {}

for fecha in fechas_reales:
    activos = select_assets_period.loc[fecha]
    activos = activos[activos==1]
    resultado[fecha] = list(activos.index)

df = pd.DataFrame({"fecha": resultado.keys(), "activos": [",".join(v) for v in resultado.values()]})

# Guardamos el resultado en un archivo csv
df.to_csv("activos_seleccionados.csv", index=False)